In [4]:
import numpy as np
# 1. GİRİŞ ÜYELİK FONKSİYONLARI

# BOY (150 - 200 cm)
def kisa_boy(x):
    if x <= 160: return 1.0
    elif x <= 170: return (170 - x) / 10
    else: return 0.0

def orta_boy(x):
    if x <= 160: return 0.0
    elif x <= 175: return (x - 160) / 15
    elif x <= 190: return (190 - x) / 15
    else: return 0.0

def uzun_boy(x):
    if x <= 180: return 0.0
    elif x <= 190: return (x - 180) / 10
    else: return 1.0

# GELİR (0 - 50000 TL)
def dusuk_gelir(x):
    if x <= 10000: return 1.0
    elif x <= 20000: return (20000 - x) / 10000
    else: return 0.0

def orta_gelir(x):
    if x <= 10000: return 0.0
    elif x <= 25000: return (x - 10000) / 15000
    elif x <= 35000: return (35000 - x) / 10000
    else: return 0.0

def yuksek_gelir(x):
    if x <= 30000: return 0.0
    elif x <= 40000: return (x - 30000) / 10000
    else: return 1.0


# 2. ÇIKIŞ ÜYELİK FONKSİYONLARI
# YAŞAM STANDARDI SKORU (0 - 100)
def dusuk_skor(x):
    if x <= 20:   return 1.0
    elif x <= 40: return (40 - x) / 20
    else:         return 0.0

def orta_skor(x):
    if x <= 30:   return 0.0
    elif x <= 50: return (x - 30) / 20
    elif x <= 70: return (70 - x) / 20
    else:         return 0.0

def yuksek_skor(x):
    if x <= 60:   return 0.0
    elif x <= 80: return (x - 60) / 20
    else:         return 1.0


# 3. KULLANICI GİRİŞİ
print("="*60)
print(" BULANIK ÇIKARIM SİSTEMİ (MİN-MAX YÖNTEMİ) ")
print("="*60)

try:
    boy_giris   = float(input("Boy değerini giriniz (cm) [150-200]    : "))
    gelir_giris = float(input("Aylık gelir giriniz (TL) [0-50000]   : "))
except ValueError:
    print("Hatalı giriş! Varsayılan değerler atanıyor (Boy: 172, Gelir: 22000)")
    boy_giris = 172.0
    gelir_giris = 22000.0

# 4. BULANIKLAŞTIRMA AŞAMASI
bulanik_boy = {
    "kisa": kisa_boy(boy_giris),
    "orta": orta_boy(boy_giris),
    "uzun": uzun_boy(boy_giris)
}

bulanik_gelir = {
    "dusuk":  dusuk_gelir(gelir_giris),
    "orta":   orta_gelir(gelir_giris),
    "yuksek": yuksek_gelir(gelir_giris)
}

print("\n--- 1. Bulanıklaştırma Sonuçları ---")
print("BOY Üyelik Dereceleri:")
for kume, derece in bulanik_boy.items():
    if derece > 0: print(f"  - {kume.capitalize():<6} : {derece:.4f}")

print("\nGELİR Üyelik Dereceleri:")
for kume, derece in bulanik_gelir.items():
    if derece > 0: print(f"  - {kume.capitalize():<6} : {derece:.4f}")


# 5. KURAL TABANI VE ÇIKARIM (MİN-MAX)

kural_tablosu = {
    ("kisa", "dusuk"):  "dusuk_skor",
    ("kisa", "orta"):   "dusuk_skor",
    ("kisa", "yuksek"): "orta_skor",
    ("orta", "dusuk"):  "dusuk_skor",
    ("orta", "orta"):   "orta_skor",
    ("orta", "yuksek"): "yuksek_skor",
    ("uzun", "dusuk"):  "orta_skor",
    ("uzun", "orta"):   "yuksek_skor",
    ("uzun", "yuksek"): "yuksek_skor",
}

# İsmi daha profesyonel olan mu_degerleri olarak güncelledik
mu_degerleri = {"dusuk_skor": 0.0, "orta_skor": 0.0, "yuksek_skor": 0.0}
cikis_etiketi = {"dusuk_skor": "Düşük YSS", "orta_skor": "Orta YSS", "yuksek_skor": "Yüksek YSS"}

print("\n--- 2. Çıkarım Aşaması (Min İlişkisi) ---")
for (boy_kume, gelir_kume), cikis_kume in kural_tablosu.items():
    mu_boy   = bulanik_boy[boy_kume]
    mu_gelir = bulanik_gelir[gelir_kume]

    # "Ve" bağlacı (Kesişim)
    ates = min(mu_boy, mu_gelir)

    # "Veya" bağlacı (Birleşim)
    mu_degerleri[cikis_kume] = max(mu_degerleri[cikis_kume], ates)

    if ates > 0:
        print(f"  EĞER Boy={boy_kume:<5} VE Gelir={gelir_kume:<7} İSE Çıkış={cikis_etiketi[cikis_kume]:<11} | min({mu_boy:.3f}, {mu_gelir:.3f}) = {ates:.3f}")

print("\n--- Birleştirilmiş Çıkış Kümeleri (Max İlişkisi) ---")

for kume, deger in mu_degerleri.items():
    print(f"  {cikis_etiketi[kume]:<11} Alfa Kesim Değeri (μ) : {deger:.4f}")


# 6. BERRAKLAŞTIRMA (Ağırlıklı Ortalama Yöntemi)
# Formül: u* = Σ(μ(u)*u) / Σ(μ(u))

# Her simetrik çıkış kümesinin TEPE NOKTASI değeri (u1, u2, u3)
u1 = 20.0  # Düşük YSS için merkez (u)
u2 = 50.0  # Orta YSS için merkez (u)
u3 = 80.0  # Yüksek YSS için merkez (u)

# Kümelerin üyelik dereceleri (μ1, μ2, μ3)
mu1 = mu_degerleri["dusuk_skor"]
mu2 = mu_degerleri["orta_skor"]
mu3 = mu_degerleri["yuksek_skor"]

# Ağırlıklı Ortalama Formülünün Uygulanması
pay   = (mu1 * u1) + (mu2 * u2) + (mu3 * u3)
payda = mu1 + mu2 + mu3

# Sıfıra bölünme hatasını engellemek için kontrol
if payda != 0:
    u_yildiz = pay / payda
else:
    u_yildiz = 50.0

print("\n--- 3. Berraklaştırma (Ağırlıklı Ortalama Yöntemi) ---")
print(f"  u1={u1}, u2={u2}, u3={u3} (Çıkış Kümeleri Tepe Noktaları)")
print(f"  μ1={mu1:.3f}, μ2={mu2:.3f}, μ3={mu3:.3f} (Üyelik Dereceleri)")
print(f"\n  u* = (μ1×u1 + μ2×u2 + μ3×u3) / (μ1 + μ2 + μ3)")
print(f"  u* = ({mu1:.3f}×{u1} + {mu2:.3f}×{u2} + {mu3:.3f}×{u3}) / ({mu1:.3f} + {mu2:.3f} + {mu3:.3f})")

print("\n" + "="*60)
print(f"  SONUÇ -> Berraklaştırılmış Değer (u*) = {u_yildiz:.2f} / 100")
print("="*60)

 BULANIK ÇIKARIM SİSTEMİ (MİN-MAX YÖNTEMİ) 
Boy değerini giriniz (cm) [150-200]    : 164
Aylık gelir giriniz (TL) [0-50000]   : 19734

--- 1. Bulanıklaştırma Sonuçları ---
BOY Üyelik Dereceleri:
  - Kisa   : 0.6000
  - Orta   : 0.2667

GELİR Üyelik Dereceleri:
  - Dusuk  : 0.0266
  - Orta   : 0.6489

--- 2. Çıkarım Aşaması (Min İlişkisi) ---
  EĞER Boy=kisa  VE Gelir=dusuk   İSE Çıkış=Düşük YSS   | min(0.600, 0.027) = 0.027
  EĞER Boy=kisa  VE Gelir=orta    İSE Çıkış=Düşük YSS   | min(0.600, 0.649) = 0.600
  EĞER Boy=orta  VE Gelir=dusuk   İSE Çıkış=Düşük YSS   | min(0.267, 0.027) = 0.027
  EĞER Boy=orta  VE Gelir=orta    İSE Çıkış=Orta YSS    | min(0.267, 0.649) = 0.267

--- Birleştirilmiş Çıkış Kümeleri (Max İlişkisi) ---
  Düşük YSS   Alfa Kesim Değeri (μ) : 0.6000
  Orta YSS    Alfa Kesim Değeri (μ) : 0.2667
  Yüksek YSS  Alfa Kesim Değeri (μ) : 0.0000

--- 3. Berraklaştırma (Ağırlıklı Ortalama Yöntemi) ---
  u1=20.0, u2=50.0, u3=80.0 (Çıkış Kümeleri Tepe Noktaları)
  μ1=0.600, μ2=